In [ ]:
## Importing Libraries

import os
import copy
import random

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as transforms
from torchvision import models

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.model_selection import train_test_split

print("Libraries Imported Successfully.")

In [ ]:
## Creating Random Seed

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random Seed Set Successfully.")

In [ ]:
## Setting Up GPU

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)

if device.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))


In [ ]:
## Setting Paths

DATASET_PATH = r"Dataset"

TRAIN_CSV = os.path.join(DATASET_PATH, "train.csv")
TEST_CSV = os.path.join(DATASET_PATH, "test.csv")

In [ ]:
## Loading Dataset

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Training Shape :", train_df.shape)
print("Testing Shape :", test_df.shape)

In [ ]:
## Pivoting Dataset

pivot_df = train_df.pivot(index="image_path", columns="target_name", values="target").reset_index()
display(pivot_df.head())

In [ ]:
## Creating Target Columns

TARGET_COLUMNS = ["Dry_Clover_g", "Dry_Dead_g", "Dry_Green_g", "Dry_Total_g", "GDM_g"]

In [ ]:
## Splitting Data in Training and Validating

train_data, valid_data = train_test_split(pivot_df, test_size=0.20, random_state=SEED, shuffle=True)

print("Training Images :", len(train_data))
print("Validation Images :", len(valid_data))

In [ ]:
## Creating Hyperparameters

IMAGE_HEIGHT = 256
IMAGE_WIDTH = 512

BATCH_SIZE = 8
LEARNING_RATE = 1e-4

EPOCHS = 50
NUM_WORKERS = 0

In [ ]:
## Creating Image Transformers

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_HEIGHT, IMAGE_WIDTH)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),

    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

valid_transform = transforms.Compose([
    transforms.Resize((IMAGE_HEIGHT, IMAGE_WIDTH)),

    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

In [ ]:
## Creatinfg BiomassDataset Class

class BiomassDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_path = os.path.join(DATASET_PATH, row["image_path"])
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        target = torch.tensor(row[TARGET_COLUMNS].values.astype(np.float32))

        return image, target

In [ ]:
## Settting Dataset objects

train_dataset = BiomassDataset(train_data, transform=train_transform)

valid_dataset = BiomassDataset(valid_data, transform=valid_transform)

In [ ]:
## Creating DataLoaders

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

In [ ]:
## Verifying Pipeline

images, labels = next(iter(train_loader))

print("Image Batch Shape :", images.shape)
print("Label Batch Shape :", labels.shape)

In [ ]:
## Loading Pretrained EfficientNet-B3

weights = models.EfficientNet_B3_Weights.DEFAULT
model = models.efficientnet_b3(weights = weights)

print("Pretrained EfficientNet-B3 Loaded Successfully.")

In [ ]:
## Original Classifier

print(model.classifier)

In [ ]:
## Replacing Classifier

in_features = model.classifier[1].in_features

model.classifier = nn.Sequential(
    nn.Dropout(0.30),
    nn.Linear(in_features, 512),
    nn.ReLU(),

    nn.Dropout(0.20),
    nn.Linear(512, 128),
    nn.ReLU(),

    nn.Linear(128, 5)
)

print("Regression Head Added Successfully.")

In [ ]:
## Sending Model to Device

model = model.to(device)
print("Model Loaded on", device)

In [ ]:
## Model Architecture

print(model)

In [ ]:
## Model Parameters

total_params = sum(p.numel() for p in model.parameters())

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("=" * 50)
print(f"Total Parameters      : {total_params:,}")
print(f"Trainable Parameters  : {trainable_params:,}")
print("=" * 50)

In [ ]:
## Loss Function

criterion = nn.MSELoss()
print("Loss Function : MSELoss")

In [ ]:
## Optimizer

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

print("Optimizer : AdamW")

In [ ]:
## Learning Rate Scheduler

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

print("Learning Rate Scheduler Ready.")

In [ ]:
## Forward Pass Test

model.eval()

with torch.no_grad():
    images, labels = next(iter(train_loader))
    images = images.to(device)
    outputs = model(images)

print("Input Shape  :", images.shape)
print("Output Shape :", outputs.shape)
print("Label Shape  :", labels.shape)

In [ ]:
## Training Variables

best_loss = np.inf

train_losses = []
valid_losses = []

print("Training Variables Initialized.")

In [ ]:
## Training Configuration 

print("=" * 50)
print("Model Training Configuration")
print("=" * 50)

print(f"Model                : EfficientNet-B3")
print(f"Output Features      : {len(TARGET_COLUMNS)}")
print(f"Image Size           : {IMAGE_HEIGHT} x {IMAGE_WIDTH}")
print(f"Batch Size           : {BATCH_SIZE}")
print(f"Epochs               : {EPOCHS}")
print(f"Learning Rate        : {LEARNING_RATE}")

print(f"Optimizer            : AdamW")
print(f"Loss Function        : MSELoss")
print(f"Scheduler            : ReduceLROnPlateau")


In [ ]:
## Creating Training Function

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(loader.dataset)

    return epoch_loss

In [ ]:
## Creating Validation Function

def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(loader.dataset)

    return epoch_loss

In [ ]:
## Training History

history = {"train_loss": [], "valid_loss": []}

In [ ]:
## Clarifying Best Model

best_loss = float("inf")
best_model_weights = copy.deepcopy(model.state_dict())

In [ ]:
## Creating Training Loop

print("=" * 50)
print("Training Started")
print("=" * 50)

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    valid_loss = validate_one_epoch(model, valid_loader, criterion, device)

    scheduler.step(valid_loss)

    history["train_loss"].append(train_loss)
    history["valid_loss"].append(valid_loss)

    print(
        f"Epoch [{epoch+1:02}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Validation Loss: {valid_loss:.4f}"
    )

    if valid_loss < best_loss:
        best_loss = valid_loss

        best_model_weights = copy.deepcopy(model.state_dict())
        torch.save(best_model_weights, "model.pth")

        print("Best Model Saved.")

In [ ]:
## Saving Training History

history_df = pd.DataFrame(history)
history_df.to_csv("training_history.csv", index=False)

print("Training History Saved.")

In [ ]:
print("=" * 50)
print("Model Training Summary")
print("=" * 50)

print(f"Best Validation Loss : {best_loss:.4f}")
print(f"Epochs               : {EPOCHS}")
print(f"Learning Rate        : {LEARNING_RATE}")


In [ ]:
## Generating Predictions

model.eval()

predictions = []
ground_truth = []

with torch.no_grad():

    for images, labels in valid_loader:

        images = images.to(device)

        outputs = model(images)

        predictions.extend(outputs.cpu().numpy())
        ground_truth.extend(labels.numpy())

predictions = np.array(predictions)
ground_truth = np.array(ground_truth)

print("Prediction Shape :", predictions.shape)
print("Ground Truth Shape :", ground_truth.shape)

In [ ]:
## All Evaluation Metrics

from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score)
mae = mean_absolute_error(ground_truth, predictions)
mse = mean_squared_error(ground_truth, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(ground_truth, predictions)

print("=" * 50)
print("Evaluation Results: ")
print("=" * 50)

print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")

In [ ]:
## Performance Per Target

results = []

for i, target in enumerate(TARGET_COLUMNS):
    mae = mean_absolute_error(ground_truth[:, i], predictions[:, i])
    mse = mean_squared_error(ground_truth[:, i], predictions[:, i])
    rmse = np.sqrt(mse)
    r2 = r2_score(ground_truth[:, i], predictions[:, i])

    results.append([target, mae, mse, rmse, r2])

results_df = pd.DataFrame(results, columns=["Target", "MAE", "MSE", "RMSE", "R2 Score"])

display(results_df)

In [ ]:
## Prediction Correlation Heatmap

correlation = pd.DataFrame(predictions, columns=TARGET_COLUMNS).corr()

plt.figure(figsize=(10, 5))

plt.imshow(correlation, cmap="viridis", interpolation="nearest")
plt.xticks(range(len(TARGET_COLUMNS)), TARGET_COLUMNS, rotation=45)
plt.yticks(range(len(TARGET_COLUMNS)), TARGET_COLUMNS)

plt.colorbar()
plt.title("Correlation Between Predicted Targets")
plt.tight_layout()
plt.show()

In [ ]:
## Predicted vs Actual

fig, axes = plt.subplots(2, 3, figsize=(10, 15))
axes = axes.flatten()

for i, target in enumerate(TARGET_COLUMNS):
    axes[i].scatter(ground_truth[:, i], predictions[:, i], alpha=0.7)
    minimum = min(ground_truth[:, i].min(), predictions[:, i].min())
    maximum = max(ground_truth[:, i].max(), predictions[:, i].max())

    axes[i].plot([minimum, maximum], [minimum, maximum], "r--")

    axes[i].set_title(target)
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

axes[-1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
## Residual Analysis

fig, axes = plt.subplots(2, 3, figsize=(10, 5))
axes = axes.flatten()

for i, target in enumerate(TARGET_COLUMNS):
    residuals = (ground_truth[:, i] - predictions[:, i])

    axes[i].scatter(predictions[:, i], residuals, alpha=0.7)
    axes[i].axhline(0, color="red", linestyle = "--")

    axes[i].set_title(target)
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("Residual")

axes[-1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
## Error Distribution

fig, axes = plt.subplots(2, 3, figsize=(10, 5))
axes = axes.flatten()

for i, target in enumerate(TARGET_COLUMNS):
    errors = np.abs(ground_truth[:, i] - predictions[:, i])

    axes[i].hist(errors, bins=20)
    axes[i].set_title(target)
    axes[i].set_xlabel("Absolute Error")

axes[-1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
## Saving Evaluation Metrics

results_df.to_csv("evaluation_metrics.csv", index = False)

print("Evaluation Metrics Saved Successfully.")

In [ ]:
print("=" * 60)
print("Training And Evaluation is completed.")
print("=" * 60)

print(f"Validation Samples : {len(valid_dataset)}")
print(f"Overall MAE        : {mae:.4f}")
print(f"Overall RMSE       : {rmse:.4f}")
print(f"Overall R² Score   : {r2:.4f}")

print()
print("Best Model Saved Successfully.")
print("Evaluation Completed Successfully.")
